# Snowpark ML: Feature Engineering

## Lab Introduction

In this lab we will explore the ML lifecycle in Snowpark ML.  Snowpark ML uses familiar Python frameworks such as scikit-learn, XGBoost and LightGBM to make adoption easy.  For illustrative purposes, we will use XGBoost to classify Snowbear Air customers into CHURN/NOT CHURN catetories.

The lab is broken in to 3 notebooks:
1) Exploratory Data Analysis and Feature Engineering
2) Training and Evaluating a Model
3) Publishing and Deploying a Model

## Notebook 01 - Exploratory Data Analysis and Feature Engineering

Steps:
1. Setup (preamble)
2. Feature Engineering and Exploratory Data Analysis

## 1. Setup

* Import Snowpark, Snowpark ML and other useful modules

In [ ]:
# Snowpark for Python
from snowflake.snowpark import Session
#from snowflake.snowpark.version import VERSION
from snowflake.snowpark.functions import *
from snowflake.snowpark.types import *

# Snowpark ML
import snowflake.ml.modeling.preprocessing as snowparkml
from snowflake.ml.modeling.pipeline import Pipeline
from snowflake.ml.modeling.metrics.correlation import correlation
from snowflake.ml.modeling.xgboost import XGBClassifier
from snowflake.ml.modeling.metrics import *


# General Data Science Modules
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Misc
import json
import joblib

# Warning Suppression
import warnings; warnings.simplefilter('ignore')

* Load configuration and connect to Snowflake

In [ ]:
config_dir = '/home/jovyan/.ssh'
configfile = config_dir + '/sf_config'

# Load configuration file
with open(configfile) as f:
    lines = f.readlines()
    
# Convert configuration to a properties map
props = {}
for line in lines:
    (key, value) = line.split('=')
    props.update({key.lower() : value[0:-1]})
    
# Convert the private key to a DER-encoded bytes object
from cryptography.hazmat.primitives import serialization
from cryptography.hazmat.backends import default_backend

with open(props['private_key_file'], "rb") as key:
    private_key = serialization.load_pem_private_key(
        key.read(),
        password=None,
        backend=default_backend()
    )
    
private_key_bytes = private_key.private_bytes(
    encoding=serialization.Encoding.DER,
    format=serialization.PrivateFormat.PKCS8,
    encryption_algorithm=serialization.NoEncryption()
)

# Connect to Snowflake
session = Session.builder.configs({**props, **{"private_key": private_key_bytes}}).create()

# current_user = session.get_current_user()
current_user = props['user']

## 2. Feature Engineering and Exploratory Data Analysis

* Create a Snowpark Dataframe based on the CUSTOMER_CHURN table

In [ ]:
customer_churnDF = session.sql("""
SELECT
    CHURNED::INT AS CHURNED,
    CUSTOMER_ID,
    SURNAME_MASKED,
    CREDIT_SCORE,
    UPPER(GEOGRAPHY) AS GEOGRAPHY,
    UPPER(GENDER) AS GENDER,
    AGE,
    TENURE::INT AS TENURE,
    MILEAGE_POINTS,
    NUM_OF_PRODUCTS::INT AS NUM_OF_PRODUCTS,
    HAS_AIRLINE_CREDIT_CARD,
    IS_ACTIVE_MEMBER,
    ESTIMATED_SALARY::INT AS ESTIMATED_SALARY
FROM
    data_science_db.public.customer_churn
""")

### Feature Transformations

Currently Snowflake supports the following preprocessing classes:
- `StandardScaler` Standardizes features by removing the mean and scaling to unit variance.
- `OrdinalEncoder` Encodes categorical features as an integer array.
- `MixMaxScaler`   Transforms features by scaling each feature to a given range.
- `LabelEncoder`   Encodes target labels with values between 0 and n_classes-1.
- `RobustScaler`   Scales features using statistics that are robust to outliers.
- `KBinsDiscretizer` Bin continuous data into intervals.
- `MaxAbsScaler`   Scale each feature by its maximum absolute value.
- `Normalizer`     Normalize samples individually to each row's unit norm.
- `OneHotEncoder`  Encode categorical features as a one-hot numeric array.
- `Binarizer`     Binarizes data (sets feature values to 0 or 1) according to the given threshold.
- `PolynomialFeatures` Generate polynomial and interaction features, see [sklearn.preprocessing.PolynomialFeatures](https://docs.snowflake.com/en/developer-guide/snowpark-ml/reference/1.2.0/modeling#snowflake-ml-modeling-preprocessing:~:text=Generate%20polynomial%20and%20interaction%20features%20For%20more%20details%20on%20this%20class%2C%20see%20sklearn.preprocessing.PolynomialFeatures)

For more info, check the [documentation](https://docs.snowflake.com/LIMITEDACCESS/snowflake-ml-preprocessing).

- Drop customer ID and surname;  this PII is unlikely to add any predictive value to our final model.

In [ ]:
customer_churnDF = customer_churnDF.drop('customer_id', 'surname_masked')

In [ ]:
customer_churnDF.show()

Let's normalize the numeric columns

In [ ]:
mms = snowparkml.MinMaxScaler(
    input_cols=["CREDIT_SCORE", "AGE", "TENURE", "MILEAGE_POINTS", "NUM_OF_PRODUCTS", "ESTIMATED_SALARY"],
    output_cols=["CREDIT_SCORE", "AGE", "TENURE", "MILEAGE_POINTS", "NUM_OF_PRODUCTS", "ESTIMATED_SALARY"])

customer_churnDF = mms.fit(customer_churnDF).transform(customer_churnDF)
customer_churnDF.show()

- Convert categorical features to numeric using Snowpark ML One Hot Encoder

In [ ]:
snowparkml_ohe = snowparkml.OneHotEncoder(input_cols = ["GENDER", "GEOGRAPHY", "HAS_AIRLINE_CREDIT_CARD", "IS_ACTIVE_MEMBER"],
                                          output_cols = ["GENDER", "GEOGRAPHY", "HAS_AIRLINE_CREDIT_CARD", "IS_ACTIVE_MEMBER"])
customer_churnDF = snowparkml_ohe.fit(customer_churnDF).transform(customer_churnDF)

* Drop the original categorical columns that have been One Hot Encoded

In [ ]:
customer_churnDF = customer_churnDF.drop("GEOGRAPHY", "GENDER", "HAS_AIRLINE_CREDIT_CARD", "IS_ACTIVE_MEMBER")
customer_churnDF.show()

* Visualize the Distribution of Classes in the dataset

In [ ]:
sns.countplot(data=customer_churnDF.to_pandas(), x='CHURNED')
plt.title('Churn Counts')
plt.xlabel('Churn')
plt.ylabel('Count')
plt.show()

* Visualize the Distribution of Credit Score of Customers

In [ ]:
# Distribution of the Credit Score
sns.histplot(data=customer_churnDF.to_pandas(), x='CREDIT_SCORE', kde=True, bins=10)
plt.title('Distribution of Credit Score')
plt.xlabel('Credit Score')
plt.ylabel('Frequency')
plt.show()

- Find the correlation between features and target column CHURNED using Snowpark Correlation

In [ ]:
customer_churnDF_pd = customer_churnDF.to_pandas()

In [ ]:
corr = customer_churnDF_pd.corr(numeric_only=True)

- Visualize these correlations with a heatmap to explore likely feature contributions

In [ ]:
plt.figure(figsize=(7, 7))
heatmap = sns.heatmap(corr, cmap="YlGnBu")

- Create train/test datasets using Snowpark Random Split


In [ ]:
(trainDF, testDF) = customer_churnDF.random_split([0.8, 0.2], seed=42)
(trainDF.count(), testDF.count()) # should be roughly 80/20 split

- Now we will persist the definitions of both train and test datasets as tables

In [ ]:
# Save the definition of the training dataset DF to a view                            
trainDF.write.mode("overwrite").save_as_table(current_user + '_db.public.training_data')

# Save the definition of the test dataset DF to a view
testDF.write.mode("overwrite").save_as_table(current_user + '_db.public.test_data')

In [ ]:
session.close()